<a href="https://colab.research.google.com/github/Omkar210/Statistics-and-ML/blob/main/Day11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np
import pandas as pd
# import warnings
# warnings.filterwarnings("ignore")
# pd.set_option('display.max_columns', None)
from numpy import random
from collections import Counter
from numpy.linalg import inv
from numpy.linalg import eig
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import pylab
from pylab import legend
from pylab import plot, show, title, xlabel, ylabel
import scipy
from scipy import stats
from scipy.stats import binom,poisson,norm,t,expon,f
from sklearn.model_selection import train_test_split
import statsmodels
from statsmodels import stats
from statsmodels.stats import weightstats as ssw
import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import os
# print(os.getcwd())
# os.chdir('/content/drive/MyDrive/CDAC')
# os.getcwd()

### Paired t-Test

This test is used when we have two paired samples, Avg difference between the paired values is compared against the claim value to established whether the difference is statistically signaficant or not.

**Key Word:** before, after

- sample size has to be same


In [3]:
before = np.array([12,14,10,9,8,13])
after = np.array([11.8,9.7,11.3,8.2,7.2,12.3])

# before - after
# bcuz before is greater values after values come down

# Ho avg(before - after) >= 2
# Ha avg(before - after) < 2  ===>  Left Tail Test

d = before - after
np.mean(d)

ssw.ttost_paired(before,after,low=2,upp=2)

(np.float64(0.8954696830206994),
 (np.float64(-1.4413160909218492),
  np.float64(0.8954696830206994),
  np.float64(5.0)),
 (np.float64(-1.4413160909218492),
  np.float64(0.10453031697930062),
  np.float64(5.0)))

In [8]:
scipy.stats.ttest_1samp(before - after,popmean=2,alternative='less')

TtestResult(statistic=np.float64(-1.4413160909218492), pvalue=np.float64(0.10453031697930062), df=np.int64(5))

Mathematics

In [5]:
(np.mean(d) -2)/((np.std(d,ddof=1))/6**0.5)

np.float64(-1.4413160909218492)

In [7]:
t.cdf(-1.4413160909218492,5)

np.float64(0.10453031697930062)

## one way ANOVA (ANalysis Of VAriance)

Purpose:
- This test is used when we have multiple sample (more than 2)
- This test will check all the pair of mean and compare whether they can be considered as equal or not

Null hypothesis : all the pair of means are equal

Ho mean(A) = mean(B)

- Sample are in categorical data.

- Used when the response is continuous data.

Assume:
- data for all sample should be normally distribution.
- variance of all the smaple should be equal.

In [14]:
dfA = pd.DataFrame({'Col1':np.repeat('A',7),'Col2':np.random.randint(10,30,7)})
dfB = pd.DataFrame({'Col1':np.repeat('B',9),'Col2':np.random.randint(20,40,9)})
dfC = pd.DataFrame({'Col1':np.repeat('C',8),'Col2':np.random.randint(15,30,8)})
df = pd.concat([dfA,dfB,dfC])

ANOVA Model

In [20]:
# ANOVA Table
mod1 = ols('Col2 ~ Col1',df).fit()
tbl = sm.stats.anova_lm(mod1)
tbl

,df,sum_sq,mean_sq,F,PR(>F)
Col1,2.0,367.619048,183.809524,6.799195,0.005287
Residual,21.0,567.714286,27.034014,NaN,NaN


In [22]:
# For validating which pair is getting false
print(pairwise_tukeyhsd(df['Col2'],df['Col1']))

 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower    upper  reject
-----------------------------------------------------
     A      B   8.7619 0.0083   2.1573 15.3665   True
     A      C   1.4286 0.8571  -5.3542  8.2113  False
     B      C  -7.3333 0.0222 -13.7015 -0.9652   True
-----------------------------------------------------


## ANOVA CALCULATION

In [41]:
df = pd.read_excel('data_anova.xlsx')
df.head()

,Col1,Col2
0,A,10
1,A,23
2,A,14
3,A,25
4,A,22


In [46]:
grp1 = df.groupby('Col1')

sample_mean = grp1.agg('mean')

grand_mean = df['Col2'].mean()

# Calculating "between" value
7*(sample_mean.loc['A'] - grand_mean)**2 + 9*(sample_mean.loc['B'] - grand_mean)**2 + 8*(sample_mean.loc['C'] - grand_mean)**2

sampA = df.Col2[np.where(df.Col1=='A')[0]]
sampB = df.Col2[np.where(df.Col1=='B')[0]]
sampC = df.Col2[np.where(df.Col1=='C')[0]]

sum((np.array(sampA) - np.array(sample_mean.loc['A']))**2) + sum((np.array(sampB) - np.array(sample_mean.loc['B']))**2) + sum((np.array(sampC) - np.array(sample_mean.loc['C']))**2)

np.float64(684.484126984127)

In [49]:
# ANOVA Table
mod1 = ols('Col2 ~ Col1',df).fit()
tbl = sm.stats.anova_lm(mod1)
tbl

,df,sum_sq,mean_sq,F,PR(>F)
Col1,2.0,842.849206,421.424603,12.929323,0.000219
Residual,21.0,684.484127,32.594482,NaN,NaN


F test statistic : this always ratio of two variance

Formula:

F = variable between sample / variable within sample

Variation within sample : Deviation of each data point from its sample mean

Variable between sample : Deviation of both/all sample with Grand mean

As the value test statistics increase it indicates that we are going away from the expected result. And so the p-value will decrease.

If p-value is small there is the very little possiblity that will happen will be less.

# Multiple ANOVA

Defination:

If the change level of factor has a different impact on the levels of other factor then we say two factors are interacting.

Change of the level Weather from dry to rain has a different impact on Factor car and Bike so we can say weather and mode are interact each other.

In [51]:
zoo = [34,43,57,40,85,68,67,53,41,24,42,52]
supp=[1,1,1,1,2,2,2,2,3,3,3,3]
lake=["R","D","R","D","R","D","R","D","R","D","R","D"]
df=pd.DataFrame({'Resp':zoo,"Factor1":supp,"Factor2":lake})
df.head()

,Resp,Factor1,Factor2
0,34,1,R
1,43,1,D
2,57,1,R
3,40,1,D
4,85,2,R


In [52]:
df.nunique()

,0
Resp,12
Factor1,3
Factor2,2


In [53]:
model = ols('Resp~Factor1+Factor2',df).fit()
res = sm.stats.anova_lm(model)
res

,df,sum_sq,mean_sq,F,PR(>F)
Factor2,1.0,176.333333,176.333333,0.543765,0.479659
Factor1,1.0,28.125000,28.125000,0.086730,0.775056
Residual,9.0,2918.541667,324.282407,NaN,NaN
